# DATALUS POC Quickstart: SIH-SUS Generative Pipeline

This notebook demonstrates the **DATALUS** framework for synthetic tabular data generation. 
We use a real-world sample of Brazilian health data (SIH-SUS) to showcase the entire pipeline: 
from raw data ingestion to ONNX edge export, including auditing and generative operations.

**DATALUS** is designed for high-dimensional, heterogeneous government datasets, ensuring privacy (LGPD alignment) 
and utility through advanced diffusion models.

## 1. Environment Setup
Detecting environment and installing dependencies if necessary.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_poc'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_poc'
    return './datalus_poc'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Install dependencies if in Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !pip install -e '.[dev]' pysus polars matplotlib
    sys.path.append(os.getcwd())


## 2. Data Ingestion (SIH-SUS)
We download a small sample of Hospital Information System (SIH) data for São Paulo (SP) using the `pysus` API.

In [ ]:
from pysus.api import sih
import polars as pl
import numpy as np

print('Downloading SIH-SUS sample...')
# Fetching SP, Jan 2024 sample (small slice for POC)
df_pandas = sih(state='SP', year=2024, month=[1])
df_polars = pl.from_pandas(df_pandas)

# Clean MORTE column (Target: 1 for death, 0 for survival)
# In SIH-SUS, MORTE=1 is death. We ensure it is integer.
df_polars = df_polars.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

# Drop strictly empty columns or identifiers to focus on modeling
cols_to_keep = [c for c in df_polars.columns if not c.startswith(('N_AIH', 'IDENT'))]
df_polars = df_polars.select(cols_to_keep).head(10000) # Small sample for speed

raw_path = f'{WORKING_DIR}/raw_sih.parquet'
df_polars.write_parquet(raw_path)
print(f'Data saved to {raw_path} | Shape: {df_polars.shape}')

## 3. Ingesting with DATALUS CLI
The `ingest` command infers the schema, detects delimiters, and prepares the data for modeling.

In [ ]:
!datalus ingest {WORKING_DIR}/raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Training the Diffusion Model
We train the residual MLP denoiser. For this POC, we use a very low number of epochs.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 3 --batch-size 1024 --gpu 0

## 5. Ab-initio Generation (Sampling)
Generating entirely new synthetic records from the learned distribution.

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 5000

## 6. Data Augmentation
Appending synthetic rows to a small existing dataset.

In [ ]:
import polars as pl
synthetic = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')
seed_for_augment = synthetic.sample(n=1000)
seed_for_augment.write_parquet(f'{WORKING_DIR}/seed_for_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_for_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 7. Minority Class Balancing
Generating records to balance specific target classes (e.g., mortality).

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 2000}'

## 8. Tabular Inpainting (RePaint)
Filling missing values (nulls) while preserving observed fields.

In [ ]:
import polars as pl
processed = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
# Create missing values in 'IDADE' column for demonstration
incomplete = processed.head(100).with_columns(
    pl.when(pl.Series(np.random.rand(100) > 0.5)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 9. Counterfactual Modification
Applying interventions (e.g., changing 'SEXO') and regenerating compatible fields.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'

## 10. Autonomous Privacy & Utility Audit
Evaluating DCR metrics and TSTR/TRTR utility ratios.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode ci_lite

## 11. Visualizing Audit Results


In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

# Plot Utility (TSTR vs TRTR)
if 'utility' in report and 'target_class_probs' in report['utility']:
    probs = report['utility']['target_class_probs']
    labels = list(probs.keys())
    real_probs = [probs[l]['real'] for l in labels]
    synth_probs = [probs[l]['synthetic'] for l in labels]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots()
    ax.bar(x - width/2, real_probs, width, label='Real')
    ax.bar(x + width/2, synth_probs, width, label='Synthetic')
    
    ax.set_ylabel('Probability')
    ax.set_title('Target Class Distribution Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.show()

## 12. Exporting to ONNX Edge
Exporting weights for browser-local or edge inference.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize

## 13. Serving and Web Deployment
Artifacts can be served via FastAPI:
`datalus serve {WORKING_DIR}/artifacts --host 0.0.0.0 --port 8000`

This enables the React WASM frontend to perform inference locally in the user's browser.